# Replace Mode

Four replacement strategies compared side by side:
- **Redact** — remove entity, leave a marker
- **Annotate** — tag entity with its label (no removal)
- **Hash** — deterministic hash token
- **Substitute** — LLM-generated synthetic values

In [13]:
import pandas as pd
df = pd.read_csv('~/Downloads/NVIDIA_synthetic_biographies.csv')

In [18]:
from pprint import pprint
pprint(df.loc[0].values[0])

('Bobby Watford, a 40‑year‑old Mexican veterinarian living in Denver, '
 'Colorado, grew up on the outskirts of the city and developed a love for '
 'animals early on. After graduating from Jefferson High, he earned his DVM at '
 'the University of Colorado Boulder, where he also completed a research stint '
 'in wildlife health. Fluent in English, Bobby has always described his '
 'upbringing as a blend of small‑town curiosity and the vibrant culture of his '
 'community, values that continue to shape his compassionate approach to '
 'animal care.\n'
 '\n'
 'Since finishing his training, Bobby has worked at VCA Animal Hospital and '
 'later at the Colorado Veterinary Clinic, where he now leads a busy '
 'mixed‑practice team. He identifies as a Christian Democrat and often '
 'volunteers at local shelters, a habit encouraged by his wife, Maya, and '
 'their two teenage children, Aria and Leo. Outside the clinic, Bobby enjoys '
 'hiking the Rockies with his family and mentoring veterina

## Setup

In [1]:
import pandas as pd
substitute_preview = pd.read_csv('/Users/lramaswamy/Documents/github/Anonymizer/docs/notebooks/crowdstrike_trace_dataframe.csv')

In [2]:
substitute_preview.drop_duplicates(subset=['_anonymizer_record_id'])[['query']].to_csv('crowdstrike_trace_dataframe_dedup.csv')

In [1]:
from pathlib import Path

from anonymizer import LoggingConfig, configure_logging

configure_logging(LoggingConfig.verbose())

try:
    NOTEBOOK_SOURCE_DIR = Path(__file__).resolve().parent
except NameError:
    # Running as .ipynb — cwd is docs/notebooks/, data lives in docs/notebook_source/
    NOTEBOOK_SOURCE_DIR = Path.cwd().parent / "notebook_source"

from anonymizer import Annotate, Anonymizer, AnonymizerConfig, AnonymizerInput, Detect, Hash, Redact, Substitute

In [2]:
anonymizer = Anonymizer()

[15:38:07] [INFO] 🔧 Anonymizer initialized with 3 model configs
[15:38:07] [INFO]   |-- 🔎 detector:  gliner-pii-detector
[15:38:07] [INFO]   |-- ✅ validator: gpt-oss-120b
[15:38:07] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## Input data

In [3]:
input_data = AnonymizerInput(
    source = "/Users/lramaswamy/Documents/github/Anonymizer/docs/notebooks/crowdstrike_trace_dataframe_dedup.csv"
    # source=str(NOTEBOOK_SOURCE_DIR / "data" / "synth_bios_sample10.csv"),
   , text_column="query",
    data_summary="crowdstrike falcon data",
    # source="/Users/lramaswamy/Documents/gitlab/synthetic-rewriting/evaluation/datasets/patient_events_first25.csv",
    # text_column="notes",
    # data_summary="these are notes on patients, diagnosis and treatment specifics are not sensitive, but exact dosage etc are",
)

## Redact

Default: `[REDACTED_FIRST_NAME]`. Customize with `format_template`.

In [4]:
pd.read_csv( "/Users/lramaswamy/Documents/github/Anonymizer/docs/notebooks/crowdstrike_trace_dataframe_dedup.csv").shape

NameError: name 'pd' is not defined

In [5]:
import os

os.environ["NVIDIA_API_KEY"] = "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"

In [6]:
LABELS = ["email", "file_path", "ipv4", "api_key", "url", "ipv4_subnet", "filename", "unique_id", "password"]
# LABELS=['first_name', 'last_name']
redact_config = AnonymizerConfig(
    detect=Detect(entity_labels=LABELS),
    replace=Substitute(),
)

redact_preview = anonymizer.preview(
    config=redact_config,
    data=input_data,
)
redact_preview.display_record(0)

[15:38:09] [INFO] 📂 Loaded 13 records from /Users/lramaswamy/Documents/github/Anonymizer/docs/notebooks/crowdstrike_trace_dataframe_dedup.csv (column: 'query')
[15:38:09] [INFO] detection labels in scope: ['api_key', 'email', 'file_path', 'filename', 'ipv4', 'ipv4_subnet', 'password', 'unique_id', 'url']
[15:38:09] [INFO]   |-- 👀 Preview mode: processing 10 of 13 records
[15:38:09] [INFO] 🔍 Running entity detection on 10 records
[15:38:09] [INFO] 🔁 Preview generation in progress


[15:38:14] [INFO] ✅ Validation passed
[15:38:14] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[15:38:14] [INFO] 🩺 Running health checks for models...
[15:38:14] [INFO]   |-- 👀 Checking 'nvidia/gliner-pii' in provider named 'nvidia' for model alias 'gliner-pii-detector'...
[15:38:15] [INFO]   |-- ✅ Passed!
[15:38:15] [INFO]   |-- 👀 Checking 'openai/gpt-oss-120b' in provider named 'nvidia' for model alias 'gpt-oss-120b'...
[15:38:16] [INFO]   |-- ✅ Passed!
[15:38:16] [INFO] 🌱 Sampling 10 records from seed dataset
[15:38:16] [INFO]   |-- seed dataset size: 13 records
[15:38:16] [INFO]   |-- sampling strategy: ordered
[15:38:16] [INFO] 📝 llm-text model config for column '_raw_detected_entities'
[15:38:16] [INFO]   |-- model: 'nvidia/gliner-pii'
[15:38:16] [INFO]   |-- model alias: 'gliner-pii-detector'
[15:38:16] [INFO]   |-- model provider: 'nvidia'
[15:38:16] [INFO]   |-- inference parameters:
[15:38:16] [INFO]   |  |-- generation_type=chat-completion
[15:38:16] [INFO] 

Original,Label,Replacement
1e0afc34e9634710bfcd2f46d8564ab1,unique_id,9f2b7c1e5a6d8349b0c3e7f1d2a4b6c8


In [7]:
redact_preview.dataframe.shape

(15, 5)

In [8]:
redact_preview.display_record(7)

Original,Label,Replacement
10.0.0.0/8,ipv4_subnet,172.16.0.0/12
224.0.0.0/4,ipv4_subnet,100.64.0.0/10


In [ ]:
redact_preview.trace_dataframe['_anonymizer_record_id'].value_counts()

0    1a2968b508b25e25b1846a2cd21b21e5
1    1b648cb2236654139262f514ac59a6be
2    a1d0a20e51fc55548b9f3d9a2e97ca5a
3    1fd54f3a404751ea91912542ef9e5356
4    7dab8cbb2f8b5b24b3f01e1dbc21b0f5
5    eb56b1ed171b504f9e4e9ee2675ef560
6    7686a6508428527f9f761289dfd79ca0
7    c13c94d8dd8e5ccd9919bcb2ba8befbe
8    2fdf7185af455ea99e207f2e779ee2bd
9    0bfd2c78a5e5511abfbd455e2233f923
Name: _anonymizer_record_id, dtype: object

In [13]:
redact_preview.trace_dataframe.final_entities.apply(lambda x: set([a['label'] for a in x['entities']]))

0    {first_name, last_name}
1    {first_name, last_name}
2    {first_name, last_name}
3    {first_name, last_name}
4    {first_name, last_name}
5    {first_name, last_name}
6    {first_name, last_name}
7    {first_name, last_name}
8    {first_name, last_name}
9    {first_name, last_name}
Name: final_entities, dtype: object

In [8]:
# redact_preview.blacklist_labels / redact_preview.whitelist_labels

In [9]:
redact_preview.trace_dataframe

,occupation,bio,_anonymizer_record_id,_raw_detected_entities,_seed_entities,_initial_tagged_text,_seed_entities_json,_tag_notation,_merged_tagged_text,_validation_candidates,...,_merged_entities,_detected_entities,bio_with_spans,_entities_by_value,final_entities,_entity_examples,_entities_for_replace,_entities_for_replace_json,_replacement_map,bio_replaced
0,bartender,"Mara Delgado, a 34‑year‑old bartender, has bec...",1a2968b508b25e25b1846a2cd21b21e5,"{""total_entities"": 16, ""entities"": [{""text"": ""...","{'entities': [{'end_position': 4, 'id': 'first...",<first_name>Mara</first_name> <last_name>Delga...,"[{""id"": ""first_name_0_4"", ""value"": ""Mara"", ""la...",xml,<first_name>Mara</first_name> <last_name>Delga...,"{'candidates': [{'context_after': ' Delgado, a...",...,"{'entities': [{'end_position': 4, 'id': 'first...","{'entities': [{'end_position': 4, 'id': 'first...",<first_name>Mara</first_name> <last_name>Delga...,{'entities_by_value': [{'labels': ['last_name'...,"{'entities': [{'end_position': 4, 'id': 'first...","{""first_name"": ""(e.g. Michael, Isabella, Carlo...","[{'labels': ['last_name'], 'labels_str': 'last...","[{""value"": ""Asheville"", ""labels"": [""last_name""...","{'replacements': [{'original': 'Asheville', 'l...","Elena Morales, a 34‑year‑old bartender, has be..."
1,doctor,"Dr. Maya Patel, 38, is a pediatric cardiologis...",1b648cb2236654139262f514ac59a6be,"{""total_entities"": 12, ""entities"": [{""text"": ""...","{'entities': [{'end_position': 8, 'id': 'first...",Dr. <first_name>Maya</first_name> <last_name>P...,"[{""id"": ""first_name_4_8"", ""value"": ""Maya"", ""la...",xml,Dr. <first_name>Maya</first_name> <last_name>P...,"{'candidates': [{'context_after': ' Patel, 38,...",...,"{'entities': [{'end_position': 8, 'id': 'first...","{'entities': [{'end_position': 8, 'id': 'first...",Dr. <first_name>Maya</first_name> <last_name>P...,{'entities_by_value': [{'labels': ['first_name...,"{'entities': [{'end_position': 8, 'id': 'first...","{""first_name"": ""(e.g. Michael, Isabella, Carlo...","[{'labels': ['first_name'], 'labels_str': 'fir...","[{""value"": ""Arjun"", ""labels"": [""first_name""], ...","{'replacements': [{'original': 'Maya', 'label'...","Dr. Anika Mehta, 38, is a pediatric cardiologi..."
2,film director,"Mira Alvarez, 38, is an independent film direc...",a1d0a20e51fc55548b9f3d9a2e97ca5a,"{""total_entities"": 14, ""entities"": [{""text"": ""...","{'entities': [{'end_position': 4, 'id': 'first...",<first_name>Mira</first_name> <last_name>Alvar...,"[{""id"": ""first_name_0_4"", ""value"": ""Mira"", ""la...",xml,<first_name>Mira</first_name> <last_name>Alvar...,"{'candidates': [{'context_after': ' Alvarez, 3...",...,"{'entities': [{'end_position': 4, 'id': 'first...","{'entities': [{'end_position': 4, 'id': 'first...",<first_name>Mira</first_name> <last_name>Alvar...,{'entities_by_value': [{'labels': ['last_name'...,"{'entities': [{'end_position': 4, 'id': 'first...","{""first_name"": ""(e.g. Michael, Isabella, Carlo...","[{'labels': ['last_name'], 'labels_str': 'last...","[{""value"": ""Alvarez"", ""labels"": [""last_name""],...","{'replacements': [{'original': 'Alvarez', 'lab...","Celia Navarro, 38, is an independent film dire..."
3,pilot,"James “Jim” Alvarez, 34, has been soaring the ...",1fd54f3a404751ea91912542ef9e5356,"{""total_entities"": 17, ""entities"": [{""text"": ""...","{'entities': [{'end_position': 5, 'id': 'first...",<first_name>James</first_name> “<first_name>Ji...,"[{""id"": ""first_name_0_5"", ""value"": ""James"", ""l...",xml,<first_name>James</first_name> “<first_name>Ji...,{'candidates': [{'context_after': ' “Jim” Alva...,...,"{'entities': [{'end_position': 5, 'id': 'first...","{'entities': [{'end_position': 5, 'id': 'first...",<first_name>James</first_name> “<first_name>Ji...,{'entities_by_value': [{'labels': ['last_name'...,"{'entities': [{'end_position': 5, 'id': 'first...","{""first_name"": ""(e.g. Michael, Isabella, Carlo...","[{'labels': ['last_name'], 'labels_st

In [10]:
redact_run = anonymizer.run(
    config=redact_config,
    data=input_data,
)

print(redact_run)
redact_run.display_record(0)

[13:35:50] [INFO] 📂 Loaded 10 records from /Users/lramaswamy/Documents/github/Anonymizer/docs/notebook_source/data/synth_bios_sample10.csv (column: 'bio')
[13:35:50] [INFO] detection labels in scope: ['first_name', 'last_name']
[13:35:50] [INFO] 🔍 Running entity detection on 10 records
[13:35:50] [INFO] 🎨 Creating Data Designer dataset
[13:35:50] [INFO] ✅ Validation passed
[13:35:50] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[13:35:50] [INFO] 🩺 Running health checks for models...
[13:35:50] [INFO]   |-- 👀 Checking 'openai/gpt-oss-120b' in provider named 'nvidia' for model alias 'gpt-oss-120b'...
[13:35:51] [INFO]   |-- ✅ Passed!
[13:35:51] [INFO]   |-- 👀 Checking 'nvidia/gliner-pii' in provider named 'nvidia' for model alias 'gliner-pii-detector'...
[13:35:51] [INFO]   |-- ✅ Passed!
[13:35:51] [INFO] ⏳ Processing batch 1 of 1
[13:35:51] [INFO] 🌱 Sampling 10 records from seed dataset
[13:35:51] [INFO]   |-- seed dataset size: 10 records
[13:35:51] [INFO]   |-- sampl

AnonymizerResult(rows=8, columns=5, trace_columns=22, failed_records=2)


Original,Label,Replacement
Asheville,last_name,Burlington
Delgado,last_name,Mendoza
Luis,first_name,Diego
Mara,first_name,Camila
Rosa,first_name,Isabel


### Custom template

`format_template="***"` replaces every entity with the same constant.

In [11]:
custom_config = AnonymizerConfig(replace=Redact(format_template="***"))

custom_preview = anonymizer.preview(
    config=custom_config,
    data=input_data,
    num_records=3,
)

custom_preview.display_record(0)

[13:36:31] [INFO] 📂 Loaded 10 records from /Users/lramaswamy/Documents/github/Anonymizer/docs/notebook_source/data/synth_bios_sample10.csv (column: 'bio')
[13:36:31] [INFO] detection labels in scope: (default: 55 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)
[13:36:31] [INFO]   |-- 👀 Preview mode: processing 3 of 10 records
[13:36:31] [INFO] 🔍 Running entity detection on 3 records
[13:36:31] [INFO] 🧐 Preview generation in progress
[13:36:31] [INFO] ✅ Validation passed
[13:36:31] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[13:36:31] [INFO] 🩺 Running health checks for models...
[13:36:31] [INFO]   |-- 👀 Checking 'openai/gpt-oss-120b' in provider named 'nvidia' for model alias 'gpt-oss-120b'...
[13:36:32] [INFO]   |-- ✅ Passed!
[13:36:32] [INFO]   |-- 👀 Checking 'nvidia/gliner-pii' in provider named 'nvidia' for model alias 'gliner-pii-detector'...
[13:36:32] [INFO]   |-- ✅ Passed!
[13:36:32] [INFO] 🌱 Sampling 3 records from seed dataset
[13:36:32] [INFO]   |-

KeyboardInterrupt: 

## Annotate

Default: `<Alice, first_name>`. Customize with `format_template` — must include `{text}` and `{label}`.

In [ ]:
annotate_config = AnonymizerConfig(replace=Annotate())

annotate_preview = anonymizer.preview(
    config=annotate_config,
    data=input_data,
    num_records=3,
)

annotate_preview.display_record(0)

## Hash

Deterministic — same input always produces the same hash. Customize `format_template`
(must include `{digest}`), `algorithm` (`sha256`/`sha1`/`md5`), and `digest_length` (6–64).

In [ ]:
hash_config = AnonymizerConfig(replace=Hash())

hash_preview = anonymizer.preview(
    config=hash_config,
    data=input_data,
    num_records=3,
)

hash_preview.display_record(0)

## Substitute

Uses an LLM to generate contextually appropriate synthetic replacements. Unlike the
strategies above, the LLM considers the full document context — matching names to emails,
cities to states, etc.

In [ ]:
substitute_config = AnonymizerConfig(replace=Substitute())

substitute_preview = anonymizer.preview(
    config=substitute_config,
    data=input_data,
    num_records=3,
)

In [ ]:
substitute_preview.display_record(0)